# Interrupt + Human Approval — Pause for High-Risk Actions
## Stop Before High-Risk Actions — Interrupt and Resume Pattern
⏱ ~10 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/79-interrupt-human-approval/interrupt_human_approval_workbook.ipynb)

Uses LangGraph interrupt() to pause before executing risky actions. The graph stops at await_approval, serializes state to MemorySaver, and resumes when you call app.invoke() with Command(resume=True/False).

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why HITL matters for irreversible actions; interrupt vs breakpoint |
| 2 | **interrupt()** — Raises GraphInterrupt — graph suspends, state saved to checkpointer |
| 3 | **MemorySaver** — In-process checkpointer that holds state across invoke calls |
| 4 | **Resume** — Command(resume=True) approves; Command(resume=False) rejects |
| 5 | **Full Demo** — 5 actions (low/medium/high risk) with approval flow |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

In [ ]:
from typing import TypedDict
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

SAMPLE_ACTIONS = [
    {"action": "Send welcome email to new user",        "risk": "low"},
    {"action": "Archive Q1 reports",                    "risk": "low"},
    {"action": "Reset user password",                   "risk": "medium"},
    {"action": "Delete inactive accounts older than 1 year", "risk": "high"},
    {"action": "Drop all temp tables in production DB", "risk": "high"},
]

# risk in state lets downstream logic auto-approve low-risk or route high-risk to a human queue
class ApprovalState(TypedDict):
    action: str; risk: str; approved: bool; result: str

# propose_action just echoes state — useful as a named node for logging/observability before the interrupt
def propose_action(state):
    return {"action": state["action"], "risk": state["risk"]}

# await_approval is the HITL node — interrupt() serializes here; graph resumes when Command(resume=...) is sent
def await_approval(state):
    decision = interrupt({"action": state["action"], "risk": state["risk"], "question": "Approve?"})
    if decision:
        return {"approved": True,  "result": f"EXECUTED: {state['action']}"}
    return  {"approved": False, "result": f"REJECTED: {state['action']}"}

g = StateGraph(ApprovalState)
g.add_node("propose_action", propose_action)
g.add_node("await_approval", await_approval)
g.add_edge(START, "propose_action")
g.add_edge("propose_action", "await_approval")
g.add_edge("await_approval", END)
# MemorySaver is an in-process checkpointer — replace with SqliteSaver or RedisSaver for cross-process persistence
checkpointer = MemorySaver()
# interrupt_before=['await_approval'] suspends before the node; interrupt_after suspends after — both require a checkpointer
app = g.compile(checkpointer=checkpointer, interrupt_before=["await_approval"])

# auto-approve dict simulates a risk policy — in production replace with a human UI event or async approval webhook
AUTO_APPROVE = {"low": True, "medium": True, "high": False}

for i, action_def in enumerate(SAMPLE_ACTIONS):
    cfg = {"configurable": {"thread_id": f"action-{i}"}}
    init = {**action_def, "approved": False, "result": ""}

    snapshot = app.invoke(init, config=cfg)

    decision = AUTO_APPROVE.get(action_def["risk"], False)
    print(f"[{action_def['risk'].upper():6s}] {action_def['action']}")
    print(f"  Auto-decision: {'APPROVE' if decision else 'REJECT'}")

    final = app.invoke(Command(resume=decision), config=cfg)
    print(f"  Result: {final['result']}")
    print()

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*